In [0]:
from pyspark.sql.functions import hour, col, date_format, when, unix_timestamp

In [0]:
df = spark.table("pysparkdbt_taxiproject.bronze.bronze_yellowtaxi")

In [0]:
df = df.withColumn('pickup_hour', hour(col("tpep_pickup_datetime")))
## dedup
df = df.dropDuplicates()

## Feature Engineering — Derived Columns

In [0]:
df = df.withColumn("pickup_day_of_week", date_format(col("tpep_pickup_datetime"), "EEEE"))

In [0]:
df = df.withColumn(
    "is_weekend",
    when(col("pickup_day_of_week").isin("Saturday", "Sunday"), True)
    .otherwise(False)
)

In [0]:
df = df.withColumn("trip_duration",
        unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime")))

## Data Quality Flags

In [0]:
df = df.withColumn(
    "reject_reason",
    when(~col("year_month").isin("2025-11", "2025-12"), "bad_timestamp")
    .when(col("trip_duration") < 0, "reversed_trip")
    .when(col("trip_duration") == 0, "zero_duration")
    .when(col("trip_distance") == 0, "zero_distance")
    .when(col("trip_distance") > 100, "distance_outlier")
    .when((col("fare_amount") < 0) | (col("total_amount") < 0), "negative_fare")
    .when(col("passenger_count") > 6, "invalid_passenger")
    .otherwise("good")
)

In [0]:
display(df.groupBy("reject_reason").count().orderBy("count", ascending=False))

reject_reason,count
good,7695991
negative_fare,411159
zero_distance,258731
zero_duration,118704
reversed_trip,1437
distance_outlier,397
bad_timestamp,24
invalid_passenger,7


## Split — Clean + Rejects

In [0]:
silver_df = df.filter(col("reject_reason") == "good")

In [0]:
reject_df = df.filter(col("reject_reason") != "good")

In [0]:
silver_df = silver_df.drop("reject_reason")

In [0]:
print("silver:", silver_df.count())
print("rejects:", reject_df.count())
print("total:", silver_df.count() + reject_df.count())

silver: 7695991
rejects: 790459
total: 8486450


## Write & Verify

In [0]:
silver_df.write.format("delta") \
        .mode("overwrite") \
        .partitionBy("year_month") \
        .saveAsTable("pysparkdbt_taxiproject.silver.silver_yellowtaxi")

reject_df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("pysparkdbt_taxiproject.silver.reject_yellowtaxi")

In [0]:
print("silver:", spark.table("pysparkdbt_taxiproject.silver.silver_yellowtaxi").count())
print("rejects:", spark.table("pysparkdbt_taxiproject.silver.reject_yellowtaxi").count())

silver: 7695991
rejects: 790459
